In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate

from langchain_core.runnables import RunnablePassthrough, RunnableParallel 
from langchain_core.output_parsers import StrOutputParser

from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = ""

In [4]:
# Carregar modelos OpenAI - Embedding e Chat

embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-3.5-turbo", max_tokens=300)


In [5]:
# Carregar o PDF

pdf_link = 'lei_nacionalidade_pt.pdf'

loader = PyPDFLoader(pdf_link, extract_images=False)

pages = loader.load_and_split()


In [6]:
# Chunking

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=4000, 
    chunk_overlap=20, 
    length_function=len, 
    add_start_index=True
)

chunks = text_splitter.split_documents(pages)



In [8]:
# Salvar chunks no vector DB

vectordb = Chroma(embedding_function=embeddings_model, persist_directory="naiveDB")

In [9]:
# Carregar o DB

naive_retriever = vectordb.as_retriever(search_kwargs={"k": 10})

In [ ]:
os.environ["COHERE_API_KEY"] = ""

In [13]:
rerank = CohereRerank(model="rerank-multilingual-v3.0", top_n=3)

compressor_retriever = ContextualCompressionRetriever(
    base_compressor=rerank,
    base_retriever=naive_retriever
)

In [14]:
TEMPLATE = """
    Você é um assistente jurídico especializado em direito de nacionalidade portuguesa.
    Responda à pergunta do usuário com base nas seguintes informações extraídas de documentos legais:
    Query:
    {question}
    
    Context:
    {context}
"""

rag_prompt = ChatPromptTemplate.from_template(TEMPLATE)

In [15]:
setup_retrieval = RunnableParallel({"question": RunnablePassthrough(), "context": compressor_retriever})

output_parser = StrOutputParser()

compressor_retrieval_chain = setup_retrieval | rag_prompt | llm | output_parser

In [16]:
compressor_retrieval_chain.invoke("Quem são considerados portugueses originários?")

'De acordo com a Lei da Nacionalidade Portuguesa, são considerados portugueses originários os descendentes de portugueses que mantenham uma ligação efetiva à comunidade nacional. Isso inclui os filhos de portugueses nascidos no estrangeiro, desde que um dos pais seja português e que estes manifestem a vontade de serem considerados portugueses, bem como os netos de portugueses que tenham laços de ligação à comunidade nacional. Além disso, podem ser considerados portugueses originários os estrangeiros que tenham residido em Portugal durante um determinado período de tempo e preencham os requisitos legais para a aquisição da nacionalidade pela via da naturalização.'